# Notebook 05 — Model Evaluation
**Member 2** | Compares all 5 models: UserBased, ItemBased, SVD, ContentBased, Hybrid.
Metrics: RMSE, Precision@10. Charts saved to reports/figures/.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded.')

## 1. Load Data & Models

In [ ]:
ratings = pd.read_csv('../data/processed/ratings_clean.csv')
movies  = pd.read_csv('../data/processed/movies_clean.csv')

user_col   = 'userId'  if 'userId'  in ratings.columns else ratings.columns[0]
item_col   = 'movieId' if 'movieId' in ratings.columns else ratings.columns[1]
rating_col = 'rating'  if 'rating'  in ratings.columns else ratings.columns[2]
movie_id_col = 'movieId' if 'movieId' in movies.columns else movies.columns[0]
title_col    = 'title'   if 'title'   in movies.columns else movies.columns[1]

# Load SVD model
with open('../models/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)

# Load content model
with open('../models/content_model.pkl', 'rb') as f:
    content_model = pickle.load(f)

cosine_sim = content_model['cosine_sim']
indices    = content_model['indices']

print('Data and models loaded successfully.')
print(f'Ratings: {ratings.shape} | Movies: {movies.shape}')

## 2. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42)
print(f'Train: {train.shape} | Test: {test.shape}')

# User-item matrix from train
user_item = train.pivot_table(index=user_col, columns=item_col, values=rating_col)
user_item_filled = user_item.fillna(0)
global_mean = train[rating_col].mean()
print(f'Global mean rating: {global_mean:.4f}')

## 3. Predict Functions for Each Model

In [ ]:
# ── SVD predict ──
def predict_svd(user_id, movie_id):
    try:
        return svd_model.predict(user_id, movie_id).est
    except:
        return global_mean

# ── User-based CF predict ──
user_sim_matrix = cosine_similarity(user_item_filled)
user_sim_df = pd.DataFrame(user_sim_matrix,
                            index=user_item.index,
                            columns=user_item.index)

def predict_user_based(user_id, movie_id, k=20):
    if user_id not in user_sim_df.index or movie_id not in user_item.columns:
        return global_mean
    sim_users = user_sim_df[user_id].drop(user_id).nlargest(k)
    ratings_col = user_item[movie_id].dropna()
    common = sim_users.index.intersection(ratings_col.index)
    if len(common) == 0:
        return global_mean
    sims = sim_users[common]
    rats = ratings_col[common]
    denom = sims.abs().sum()
    return float(np.dot(sims, rats) / denom) if denom != 0 else global_mean

# ── Item-based CF predict ──
item_sim_matrix = cosine_similarity(user_item_filled.T)
item_sim_df = pd.DataFrame(item_sim_matrix,
                            index=user_item.columns,
                            columns=user_item.columns)

def predict_item_based(user_id, movie_id, k=20):
    if movie_id not in item_sim_df.index or user_id not in user_item.index:
        return global_mean
    sim_items = item_sim_df[movie_id].drop(movie_id).nlargest(k)
    user_row = user_item.loc[user_id].dropna()
    common = sim_items.index.intersection(user_row.index)
    if len(common) == 0:
        return global_mean
    sims = sim_items[common]
    rats = user_row[common]
    denom = sims.abs().sum()
    return float(np.dot(sims, rats) / denom) if denom != 0 else global_mean

# ── Content-based predict (use similarity as proxy, scale to 1-5) ──
def predict_content(user_id, movie_id):
    if user_id not in user_item.index or movie_id not in indices:
        return global_mean
    user_row = user_item.loc[user_id].dropna()
    if user_row.empty:
        return global_mean
    j = indices[movie_id]
    sims, rats = [], []
    for mid, rat in user_row.items():
        if mid in indices:
            i = indices[mid]
            sims.append(cosine_sim[i, j])
            rats.append(rat)
    if not sims or sum(sims) == 0:
        return global_mean
    sims = np.array(sims)
    rats = np.array(rats)
    return float(np.dot(sims, rats) / sims.sum())

# ── Hybrid predict ──
def predict_hybrid(user_id, movie_id, alpha=0.7):
    svd_p  = predict_svd(user_id, movie_id)
    cont_p = predict_content(user_id, movie_id)
    return alpha * svd_p + (1 - alpha) * cont_p

print('All predict functions defined.')

## 4. Compute RMSE for All 5 Models

In [ ]:
# Use a sample of test set for speed
test_sample = test.sample(min(300, len(test)), random_state=42)

models = {
    'UserBased':    predict_user_based,
    'ItemBased':    predict_item_based,
    'SVD':          predict_svd,
    'ContentBased': predict_content,
    'Hybrid':       predict_hybrid,
}

rmse_results = {}
for name, fn in models.items():
    preds = [fn(row[user_col], row[item_col]) for _, row in test_sample.iterrows()]
    actuals = test_sample[rating_col].tolist()
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    rmse_results[name] = round(rmse, 4)
    print(f'{name:15s}  RMSE = {rmse:.4f}')

## 5. Precision@10 for All 5 Models

In [ ]:
THRESHOLD = 3.5  # rating ≥ threshold = relevant
eval_users = test_sample[user_col].unique()[:30]
all_movies = ratings[item_col].unique().tolist()

def precision_at_k(user_id, predict_fn, k=10):
    rated = set(train[train[user_col] == user_id][item_col].tolist())
    candidates = [m for m in all_movies if m not in rated][:200]  # cap for speed
    if not candidates:
        return 0.0
    scores = [(m, predict_fn(user_id, m)) for m in candidates]
    scores.sort(key=lambda x: x[1], reverse=True)
    top_k = scores[:k]
    # Check how many of top-k are actually liked in test
    user_test = test[(test[user_col] == user_id) & (test[rating_col] >= THRESHOLD)]
    relevant = set(user_test[item_col].tolist())
    hits = sum(1 for m, _ in top_k if m in relevant)
    return hits / k

precision_results = {}
for name, fn in models.items():
    p_scores = [precision_at_k(u, fn, k=10) for u in eval_users]
    precision_results[name] = round(np.mean(p_scores), 4)
    print(f'{name:15s}  Precision@10 = {precision_results[name]:.4f}')

## 6. Save Charts

In [ ]:
os.makedirs('../reports/figures', exist_ok=True)
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']
model_names = list(rmse_results.keys())

# ── RMSE Chart ──
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(model_names, [rmse_results[m] for m in model_names],
              color=colors, edgecolor='black', width=0.5)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('RMSE (lower is better)', fontsize=12)
ax.set_title('RMSE Comparison — All 5 Models', fontsize=14)
for bar, v in zip(bars, rmse_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/rmse_comparison.png', dpi=150)
plt.close()
print('Saved: rmse_comparison.png')

# ── Precision Chart ──
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(model_names, [precision_results[m] for m in model_names],
              color=colors, edgecolor='black', width=0.5)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Precision@10 (higher is better)', fontsize=12)
ax.set_title('Precision@10 Comparison — All 5 Models', fontsize=14)
for bar, v in zip(bars, precision_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/precision_comparison.png', dpi=150)
plt.close()
print('Saved: precision_comparison.png')

# ── Alpha vs RMSE Chart ──
alphas = [0.3, 0.5, 0.7, 0.9]
alpha_rmse = []
for a in alphas:
    preds = [predict_hybrid(row[user_col], row[item_col], alpha=a)
             for _, row in test_sample.iterrows()]
    rmse = np.sqrt(mean_squared_error(test_sample[rating_col].tolist(), preds))
    alpha_rmse.append(round(rmse, 4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([str(a) for a in alphas], alpha_rmse,
        marker='o', linewidth=2, color='#C44E52', markersize=8)
ax.set_xlabel('Alpha (collaborative weight)', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Hybrid Model — Alpha vs RMSE', fontsize=14)
for x, y in zip([str(a) for a in alphas], alpha_rmse):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/alpha_vs_rmse.png', dpi=150)
plt.close()
print('Saved: alpha_vs_rmse.png')

## 7. Save model_comparison.csv

In [ ]:
os.makedirs('../reports', exist_ok=True)

comparison_df = pd.DataFrame({
    'Model': model_names,
    'RMSE': [rmse_results[m] for m in model_names],
    'Precision@10': [precision_results[m] for m in model_names]
})
comparison_df['Rank_RMSE'] = comparison_df['RMSE'].rank().astype(int)
comparison_df['Rank_Precision'] = comparison_df['Precision@10'].rank(ascending=False).astype(int)

comparison_df.to_csv('../reports/model_comparison.csv', index=False)

print('Saved: model_comparison.csv')
print()
print(comparison_df.to_string(index=False))